## SRP006565 / GSE28663

**paper:** [PMID: 21750260](https://pmc.ncbi.nlm.nih.gov/articles/PMC3176115/) - The genetic basis of rapidly evolving male genital morphology in Drosophila. 2011

**date, curator:** 2026-07-16, Sara Carsanaro

**notes**
* strains used -> not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype
    * Q1(A) and 3Q1(A)
    * D. mauritiana P-insertion Q1 - wild type mini-white gene, rescues eye pigmentation
* SRA says adult, however it really seems like third larval instar given genital disc tissue, also other parts of paper mention 3rd instar larva including methods, annotating as third larval instar

### annotation summary

In [25]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Genital Disc,UBERON:6001785,insect male genital disc,perfect match


In [26]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,third larval instar,UBERON:8000002,third instar larva stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP006565"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
curl: (22) The requested URL returned error: 500
>78 ERROR: >78 curl command failed ( Thu Jul 16 17:23:43 CEST 2026 ) with: 22>78
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi -d id=21750260&dbfrom=pubmed&cmd=prlinks&tool=edirect&edirect=16.2&edirect_os=Darwin&email=scarsana%40SORGE42778>78
HTTP/1.0 500 Internal Server Error
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ elink.fcgi -id 21750260 -dbfrom pubmed -cmd prlinks -tool edirect -edirect 16.2 -edirect_os Darwin -email scarsana@SORGE42778
EMPTY RESULT
SECOND ATTEMPT
curl: (22) The requested URL returned error

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,,,,M,,hybrid introgression line Q1(A),1007467,,,,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710170,,hybrid introgression line Q1(A),,,,TRANSCRIPTOMIC,cDNA
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,,,,M,,hybrid introgression line 3Q1(A),1007467,,,,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710169,,hybrid introgression line 3Q1(A),,,,TRANSCRIPTOMIC,cDNA
2,SRX059377,SRP006565,Illumina Genome Analyzer II,SRS190442,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Genital Disc']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:6001785'
library.loc[:,'anatName'] = 'insect male genital disc'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,perfect match,not documented,,M,,hybrid introgression line Q1(A),1007467,,,,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710170,,hybrid introgression line Q1(A),,,,TRANSCRIPTOMIC,cDNA
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,perfect match,not documented,,M,,hybrid introgression line 3Q1(A),1007467,,,,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710169,,hybrid introgression line 3Q1(A),,,,TRAN

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['third larval instar']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:8000002'
library.loc[:,'stageName'] = 'third instar larva stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line Q1(A),1007467,,,,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710170,,hybrid introgression line Q1(A),,,,TRANSCRIPTOMIC,cDNA
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line 3Q1(A),1007467,,,,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were p

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'MicroPoly(A)Purist Kit'
# full_length or 3'
#my_protocolType = ''

library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710170,,hybrid introgression line Q1(A),,,,TRANSCRIPTOMIC,cDNA
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line 3Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,,,,,16/07/2026,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each g

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-17'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,,,,SAC,2026-07-17,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes. Images were processed using Illumina's GenomeStudio software.",,GSM710170,,hybrid introgression line Q1(A),,,,TRANSCRIPTOMIC,cDNA
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line 3Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,,,,SAC,2026-07-17,"samples were fragmented to ~250 b.p. by chemical fragmentation. First strand cDNA was synthesized using SuperScriptÂ® II Reverse Transcriptase (Invitrogen) and a combination of random hexamer and oligo (dT) primers; second strand cDNA was synthesized using DNA polymerase I in combination with ribonuclease H. Double stranded cDNA templates were blunt ended using End-Itâ¢ Repair Kit (Epicentre), and A-overhangs were added at both ends with Klenow fragment (3'â5' exo-minus). Illumina GAII sequencing adapters were then ligated to both ends of the cDNA templates using Fast-Linkâ¢ DNA Ligation Kit (Epicentre). We then enriched for cDNA templates by performing multiplex incorporating PCR reactions and isolated 250-550 base pair fragments by gel purification. During PCR, unique index sequences (Illumina) were incorporated into each biological sample to allow identification of reads from each sample when multiple samples were sequenced on a single lane of the flow cell. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of 

#### comments

In [15]:
library.loc[:,'comment'] = 'PMID: 21750260, not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype'

#### save complete file with correct columns

In [16]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710170,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,"PMID: 21750260, not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype",,,SAC,2026-07-17
1,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710169,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line 3Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,"PMID: 21750260, not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype",,,SAC,2026-07-17
2,SRX059377,SRP006565,Illumina Genome Analyzer II,SRS190442,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710168,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,Q1,7226,MicroPoly(A)Purist Kit,,polyA,,,Maur_RNASeq,"SAMN00262254,GSM710168",,,,,,,"PMID: 21750260, not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype",,,SAC,2026-07-17
3,SRX059376,SRP006565,Illumina Genome Analyzer II,SRS190441,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM710167,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,w,7238,MicroPoly(A)Purist Kit,,polyA,,,Sech_RNASeq,"SAMN00262253,GSM710167",,,,,,,"PMID: 21750260, not 100% certain about the Q1(A)/3Q1(A) but i believe it is fine given it rescues wild type phenotype",,,SAC,2026-07-17


### experiment annotations

In [17]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP006565,The Genetic Basis of Rapidly Evolving Male Genital Morphology in Drosophila,"Deep sequencing of total RNA extracted from the genital discs of males for each of the following strains : Drosophila sechellia, Drosophila mauritiana, hybrid introgression line 3Q1(A) and hybrid introgression line Q1(A) Overall design: Analysis of poly(A)+ RNA for three independent biological replicates of sequencing libraries for each of the following strains: D. sechellia w, D. mauritiana P-insertion Q1, hybrid introgression line 3Q1(A), and hybrid introgression line Q1(A). Male genital discs were obtained as described above, and total RNA was extracted using RNAqueousÂ®-Micro Kit (Ambion). Poly(A)+ transcripts were isolated subsequently using MicroPoly(A)Puristâ¢ Kit (Ambion). To facilitate normalization of reads across our samples, at this stage of library construction we spiked-in small amounts of exogenous RNA from ArrayControlâ¢ Kit (Ambion) into each sample of poly(A)+ RNA. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes.",SRA,,,,,,GSE28663,PRJNA138869,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [18]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

4

In [19]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP006565,The Genetic Basis of Rapidly Evolving Male Genital Morphology in Drosophila,"Deep sequencing of total RNA extracted from the genital discs of males for each of the following strains : Drosophila sechellia, Drosophila mauritiana, hybrid introgression line 3Q1(A) and hybrid introgression line Q1(A) Overall design: Analysis of poly(A)+ RNA for three independent biological replicates of sequencing libraries for each of the following strains: D. sechellia w, D. mauritiana P-insertion Q1, hybrid introgression line 3Q1(A), and hybrid introgression line Q1(A). Male genital discs were obtained as described above, and total RNA was extracted using RNAqueousÂ®-Micro Kit (Ambion). Poly(A)+ transcripts were isolated subsequently using MicroPoly(A)Puristâ¢ Kit (Ambion). To facilitate normalization of reads across our samples, at this stage of library construction we spiked-in small amounts of exogenous RNA from ArrayControlâ¢ Kit (Ambion) into each sample of poly(A)+ RNA. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes.",SRA,total,Bgee 1K,4,MicroPoly(A)Purist Kit,,GSE28663,PRJNA138869,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [20]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '21750260'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3176115/'
experiment.loc[:,'DOI'] = '10.1534/genetics.111.130815'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP006565,The Genetic Basis of Rapidly Evolving Male Genital Morphology in Drosophila,"Deep sequencing of total RNA extracted from the genital discs of males for each of the following strains : Drosophila sechellia, Drosophila mauritiana, hybrid introgression line 3Q1(A) and hybrid introgression line Q1(A) Overall design: Analysis of poly(A)+ RNA for three independent biological replicates of sequencing libraries for each of the following strains: D. sechellia w, D. mauritiana P-insertion Q1, hybrid introgression line 3Q1(A), and hybrid introgression line Q1(A). Male genital discs were obtained as described above, and total RNA was extracted using RNAqueousÂ®-Micro Kit (Ambion). Poly(A)+ transcripts were isolated subsequently using MicroPoly(A)Puristâ¢ Kit (Ambion). To facilitate normalization of reads across our samples, at this stage of library construction we spiked-in small amounts of exogenous RNA from ArrayControlâ¢ Kit (Ambion) into each sample of poly(A)+ RNA. Paired-end sequencing was carried out by loading the samples onto four lanes (three samples per lane) of a flow cell and run on an Illumina Genome Analyzer IIx sequencer using 72 cycles per end of each paired-end read. Biological replicates of each genotype were loaded onto separate lanes.",SRA,total,Bgee 1K,4,MicroPoly(A)Purist Kit,,GSE28663,PRJNA138869,21750260,https://pmc.ncbi.nlm.nih.gov/articles/PMC3176115/,10.1534/genetics.111.130815,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [23]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [27]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [28]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
68224,ERX10346022,ERP145054,Illumina HiSeq 4000,ERS14651019,UBERON:6004535,insect proboscis,UBERON:0000066,fully formed stage,,proboscis and maxillary palps,adult,other,not documented,perfect match,M,K-AWA036,,28584,KAPA Stranded mRNA-Seq Kit,full_length,polyA,,,SUZ_PR_M_2,SAMEA112653308,2-10,day post eclosion,,,,,PMID: 38316749,,,SAC,2026-07-16
68225,ERX10346023,ERP145054,Illumina HiSeq 4000,ERS14651020,UBERON:6004535,insect proboscis,UBERON:0000066,fully formed stage,,proboscis and maxillary palps,adult,other,not documented,perfect match,M,K-AWA036,,28584,KAPA Stranded mRNA-Seq Kit,full_length,polyA,,,SUZ_PR_M_3,SAMEA112653309,2-10,day post eclosion,,,,,PMID: 38316749,,,SAC,2026-07-16
68226,SRX059379,SRP006565,Illumina Genome Analyzer II,SRS190444,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,3A_RNASeq,"SAMN00262256,GSM710170",,,,,,,"PMID: 21750260, not 100% certain about the Q1(...",,,SAC,2026-07-17
68227,SRX059378,SRP006565,Illumina Genome Analyzer II,SRS190443,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,hybrid introgression line 3Q1(A),1007467,MicroPoly(A)Purist Kit,,polyA,,,Q1A_RNASeq,"SAMN00262255,GSM710169",,,,,,,"PMID: 21750260, not 100% certain about the Q1(...",,,SAC,2026-07-17
68228,SRX059377,SRP006565,Illumina Genome Analyzer II,SRS190442,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,Q1,7226,MicroPoly(A)Purist Kit,,polyA,,,Maur_RNASeq,"SAMN00262254,GSM710168",,,,,,,"PMID: 21750260, not 100% certain about the Q1(...",,,SAC,2026-07-17
68229,SRX059376,SRP006565,Illumina Genome Analyzer II,SRS190441,UBERON:6001785,insect male genital disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Genital Disc,third larval instar,perfect match,not documented,perfect match,M,,w,7238,MicroPoly(A)Purist Kit,,polyA,,,Sech_RNASeq,"SAMN00262253,GSM710167",,,,,,,"PMID: 21750260, not 100% certain about the Q1(...",,,SAC,2026-07-17


In [29]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1295,SRP056962,Total RNA Sequencing of Drosophila pseudoobscu...,"Drosophila pseudoobscura, Drosophila sechellia...",SRA,total,Bgee 1K,5,TruSeq Stranded Total RNA,full_length,,PRJNA280578,25970244,https://pmc.ncbi.nlm.nih.gov/articles/PMC4529404/,10.1038/nature14475,,
1296,ERP145054,Comparative transcriptomics of chemosensory ti...,These data contain RNA-seq samples generated f...,SRA,total,Bgee 1K,147,KAPA Stranded mRNA-Seq Kit,full_length,,PRJEB60016,38316749,https://pmc.ncbi.nlm.nih.gov/articles/PMC10844...,10.1038/s41467-023-44558-4,E-MTAB-12656[ArrayExpress],
1297,SRP006565,The Genetic Basis of Rapidly Evolving Male Gen...,Deep sequencing of total RNA extracted from th...,SRA,total,Bgee 1K,4,MicroPoly(A)Purist Kit,,GSE28663,PRJNA138869,21750260,https://pmc.ncbi.nlm.nih.gov/articles/PMC3176115/,10.1534/genetics.111.130815,,


### add annotations to git

In [30]:
! git pull

Already up to date.


In [31]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [32]:
! git add $git_experiment_path $git_library_path

In [33]:
! git commit -m $commit_message_exp

[develop b37db62] adding annotated bulk experiment SRP006565
 2 files changed, 5 insertions(+)


In [34]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.87 KiB | 734.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   bec2f75..b37db62  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push